# Практическая работа №3. Самостоятельная работа: распределённый анализ собственного датасета

**Набор данных:** оптовые/розничные продажи торговой компании (каталог `archive/`): факт «Продажи» (~865 тыс. строк, 11 полей), справочники «Продукты» (~21 тыс.) и «Клиенты» (~3,8 тыс.). Источник: учебный набор из каталога `archive/`, выданный вместе с практической работой; условия использования — только в учебных целях. Данные обезличены (названия организаций и ИНН вымышленные, физические лица представлены единственной записью «Частное лицо»), персональных данных нет.

**Цель:** воспроизвести модель Spark на другой предметной области: оценить выручку, скидки, маржу и структуру продаж по филиалам, месяцам, товарным группам и клиентам.

## 1. Постановка задачи и архитектура входа
Исходные файлы — CSV с разделителем табуляции, кодировка UTF-8 с BOM, десятичная запятая, даты `dd.MM.yyyy`, пустые значения записаны строкой `NULL`. Поэтому staging из СУБД не нужен: Spark читает CSV напрямую, а типы задаются явной схемой.

Связи: `Продажи.Product_key → Продукты.Product_Key` (многие-к-одному), `Продажи.Client_ID → Клиенты.Client_ID` (многие-к-одному).

## 2. SparkSession и окружение

In [19]:
import os, sys, subprocess
from pathlib import Path

if os.name == "nt" and Path("hadoop/bin/winutils.exe").exists():
    os.environ["HADOOP_HOME"] = str(Path("hadoop").resolve())
    os.environ["PATH"] += os.pathsep + str(Path("hadoop/bin").resolve())

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Python :", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

spark = (
    SparkSession.builder
    .appName("Practice4_OwnDataset_Sales")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)
print("shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Python : 3.12.10
PySpark: 3.5.3
java version "1.8.0_503"
Master: local[*]
App: Practice4_OwnDataset_Sales
Spark: 3.5.3
shuffle.partitions: 8


## 3. Чтение основной таблицы и схема
Сначала читаем без схемы (все поля — `string`), чтобы посмотреть структуру.

In [20]:
SRC = Path("archive")

sales_raw = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("encoding", "UTF-8")
    .csv(str(SRC / "Продажи.csv"))
)

print("Строк:", sales_raw.count())
print("Столбцов:", len(sales_raw.columns))
print(sales_raw.columns)
sales_raw.printSchema()

Строк: 865222
Столбцов: 11
['Себестоимость', 'Дата покупки', 'Номер чека', 'Product_key', 'Филиал', 'Количество', 'Сумма покупки', 'Сумма скидки', 'Client_ID', 'Машина доставки', 'Адрес']
root
 |-- Себестоимость: string (nullable = true)
 |-- Дата покупки: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Product_key: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Количество: string (nullable = true)
 |-- Сумма покупки: string (nullable = true)
 |-- Сумма скидки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Машина доставки: string (nullable = true)
 |-- Адрес: string (nullable = true)



### Явная схема (StructType) и сравнение с `inferSchema`
Числа записаны с десятичной запятой (`113,33`), поэтому `inferSchema` не распознает их как числа. Задаём явную схему с латинскими именами полей: числа с запятой читаем как `string` и приводим позже; целые (`Количество`, суммы без копеек) тоже читаем строкой, чтобы не потерять строки с нестандартными значениями.

In [21]:
from pyspark.sql.types import StructType, StructField, StringType

sales_schema = StructType([
    StructField("cost_price", StringType(), True),
    StructField("purchase_date", StringType(), True),
    StructField("receipt_no", StringType(), True),
    StructField("product_key", StringType(), True),
    StructField("branch", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("client_id", StringType(), True),
    StructField("vehicle", StringType(), True),
    StructField("address", StringType(), True),
])

sales = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("nullValue", "NULL")
    .schema(sales_schema)
    .csv(str(SRC / "Продажи.csv"))
)

sales_infer = (
    spark.read.option("header", True).option("sep", "\t")
    .option("nullValue", "NULL").option("inferSchema", True)
    .csv(str(SRC / "Продажи.csv"))
)
print("=== inferSchema ===")
sales_infer.printSchema()
print("=== явная схема ===")
sales.printSchema()
print("Строк:", sales.count())

=== inferSchema ===
root
 |-- Себестоимость: string (nullable = true)
 |-- Дата покупки: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Product_key: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Количество: integer (nullable = true)
 |-- Сумма покупки: string (nullable = true)
 |-- Сумма скидки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Машина доставки: string (nullable = true)
 |-- Адрес: string (nullable = true)

=== явная схема ===
root
 |-- cost_price: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- vehicle: string (nullable = true)
 |-- address: string (nullable = true)

Строк: 865222


**Вывод.** Даже с `nullValue="NULL"` `inferSchema` распознал числом только `Количество`: `Себестоимость`, `Сумма покупки`, `Сумма скидки` (десятичная запятая) и даты остались `string`; кроме того, он требует дополнительного прохода по 865 тыс. строк, а без `nullValue` маркер `NULL` превратил бы в `string` и числовые поля. Явная схема даёт воспроизводимый контракт, а типы приводятся осознанно.

## 4. Transformations
Создаём типизированный DataFrame и вычисляемые признаки:
* `purchase_date` → `date`, `month`;
* числа с запятой → `double`;
* `net_amount = amount − discount` (выручка после скидки), `discount_pct`;
* `cost_total = cost_price × quantity`, `margin = net_amount − cost_total`;
* `is_delivery` — есть ли адрес доставки;
* `product_group_l1` — корневая ветка иерархии группы (добавим после JOIN).

> Допущение: поле «Себестоимость» — за единицу товара, «Сумма покупки» — сумма по строке до скидки.

In [22]:
def to_num(col):
    """'113,33' -> 113.33 (десятичная запятая)"""
    return F.regexp_replace(F.trim(F.col(col)), ",", ".").cast("double")

sales_t = (
    sales
    .withColumn("purchase_date", F.to_date("purchase_date", "dd.MM.yyyy"))
    .withColumn("month", F.date_format("purchase_date", "yyyy-MM"))
    .withColumn("branch", F.trim("branch"))
    .withColumn("cost_price", to_num("cost_price"))
    .withColumn("quantity", to_num("quantity"))
    .withColumn("amount", to_num("amount"))
    .withColumn("discount", to_num("discount"))
    .withColumn("net_amount", F.col("amount") - F.col("discount"))
    .withColumn("discount_pct", F.when(F.col("amount") > 0, F.col("discount") / F.col("amount") * 100))
    .withColumn("cost_total", F.col("cost_price") * F.col("quantity"))
    .withColumn("margin", F.col("net_amount") - F.col("cost_total"))
    .withColumn("is_delivery", F.col("address").isNotNull())
    .drop("vehicle")
)

sales_t.printSchema()
sales_t.select("purchase_date", "branch", "quantity", "amount", "discount", "net_amount",
               "cost_total", "margin", "is_delivery").show(5, truncate=False)

sales_t.agg(
    F.min("purchase_date").alias("first_date"),
    F.max("purchase_date").alias("last_date"),
    F.sum(F.col("purchase_date").isNull().cast("int")).alias("bad_dates"),
    F.sum((F.col("quantity") <= 0).cast("int")).alias("qty_le_0"),
    F.sum((F.col("amount") < 0).cast("int")).alias("amount_lt_0"),
    F.sum((F.col("margin") < 0).cast("int")).alias("margin_lt_0"),
    F.sum(F.col("margin").isNotNull().cast("int")).alias("margin_known"),
).show()

root
 |-- cost_price: double (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- client_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- month: string (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- cost_total: double (nullable = true)
 |-- margin: double (nullable = true)
 |-- is_delivery: boolean (nullable = false)

+-------------+----------------------------+--------+------+--------+----------+----------+---------+-----------+
|purchase_date|branch                      |quantity|amount|discount|net_amount|cost_total|margin   |is_delivery|
+-------------+----------------------------+--------+------+--------+----------+----------+---------+-----------

**Вывод.** Использованы `withColumn`, `select`, `drop`, `filter` (далее), `to_date`, `regexp_replace`, `cast`, `when` — более пяти разных transformations. Все даты разобраны (`bad_dates = 0`), возвратов (`quantity ≤ 0`) и отрицательных сумм нет. Около 13 % строк с известной маржей (`margin_lt_0` ≈ 103,5 тыс. из 778,7 тыс.) имеют отрицательную маржу: это либо продажи ниже себестоимости (акции, уценка), либо неточность допущения о смысле поля «Себестоимость»; это ограничение расчёта маржи (см. итоговые выводы).

## 5. Качество данных: пропуски
Считаем пропуски во всех полях, затем объясняем предметный смысл каждого.

In [23]:
null_counts = sales_t.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["cost_price", "receipt_no", "quantity", "address", "purchase_date", "amount"]
])
null_counts.show(vertical=True)

sales_t.groupBy("branch").agg(
    F.count("*").alias("rows"),
    F.sum(F.col("address").isNull().cast("int")).alias("no_address"),
    F.sum(F.col("cost_price").isNull().cast("int")).alias("no_cost"),
    F.sum(F.col("receipt_no").isNull().cast("int")).alias("no_receipt"),
    F.sum(F.col("quantity").isNull().cast("int")).alias("no_qty"),
).orderBy(F.desc("rows")).show(truncate=False)

sales_t.filter(F.col("quantity").isNull()).select(
    "branch", "receipt_no", "cost_price", "amount", "discount"
).show(5, truncate=False)

-RECORD 0---------------
 cost_price    | 86431  
 receipt_no    | 3934   
 quantity      | 100    
 address       | 832181 
 purchase_date | 0      
 amount        | 0      

+----------------------------+------+----------+-------+----------+------+
|branch                      |rows  |no_address|no_cost|no_receipt|no_qty|
+----------------------------+------+----------+-------+----------+------+
|ТЗ-2 ( Ленинградское шоссе) |431372|422993    |42858  |2807      |43    |
|ТЗ-1 ( ул.Фадеева)          |238046|218845    |23799  |19        |33    |
|ТЗ-3 (Курский)              |190730|185518    |19243  |11        |24    |
|Кафе-1 ( ул.Фадеева)        |3977  |3977      |417    |0         |0     |
|Багетная мастерская         |848   |848       |91     |848       |0     |
|РС (Распределительный склад)|249   |0         |23     |249       |0     |
+----------------------------+------+----------+-------+----------+------+

+---------------------------+----------+----------+--------+--------+
|br

Решения по пропускам:

| Поле | Смысл пропуска | Решение |
|---|---|---|
| `address` | нет доставки (самовывоз/розница) | **не удалять**; порождает признак `is_delivery` |
| `receipt_no` | продажи без чека (оптовые отгрузки, РС) | **заполнить** маркером `NO_RECEIPT` — категориальный идентификатор, числа не выдумываем |
| `cost_price` | себестоимость неизвестна | **не заполнять** нулём (исказило бы маржу); маржа считается только по строкам с известной себестоимостью через `filter(cost_price IS NOT NULL)` |
| `quantity` | количество не указано | **не заполнять**; строки остаются в выручке (`amount` известна), исключаются только из расчётов, зависящих от количества |

In [24]:
sales_c = sales_t.fillna({"receipt_no": "NO_RECEIPT"})

with_cost = sales_c.filter(F.col("cost_price").isNotNull() & F.col("quantity").isNotNull())
print("Всего строк:", sales_c.count())
print("Строк с известной себестоимостью и количеством:", with_cost.count())

Всего строк: 865222
Строк с известной себестоимостью и количеством: 778703


**Вывод.** Пропуски структурно связаны с процессом (самовывоз, розница без чека, отсутствие учёта себестоимости) — удалять строки из-за `null` было бы ошибкой. Для маржи используется явный фильтр, для чека — метка, для количества/себестоимости — без выдуманных значений.

## 6. Агрегации
### 6.1. Показатели по филиалам (6 агрегатов)

In [25]:
branch_stats = (
    sales_c
    .groupBy("branch")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("receipt_no").alias("receipts"),
        F.countDistinct("client_id").alias("clients"),
        F.round(F.sum("amount"), 0).alias("gross"),
        F.round(F.sum("discount"), 0).alias("discounts"),
        F.round(F.sum("net_amount"), 0).alias("net_revenue"),
        F.round(F.avg("discount_pct"), 2).alias("avg_discount_pct"),
    )
    .orderBy(F.desc("net_revenue"))
)
branch_stats.show(truncate=False)

+----------------------------+------+--------+-------+-----------+---------+-----------+----------------+
|branch                      |lines |receipts|clients|gross      |discounts|net_revenue|avg_discount_pct|
+----------------------------+------+--------+-------+-----------+---------+-----------+----------------+
|ТЗ-2 ( Ленинградское шоссе) |431372|78785   |406    |5.8750803E7|634368.0 |5.8116436E7|1.02            |
|ТЗ-1 ( ул.Фадеева)          |238046|46338   |1469   |4.4343168E7|1114995.0|4.3228173E7|1.35            |
|ТЗ-3 (Курский)              |190730|32297   |303    |2.6249056E7|498665.0 |2.5750391E7|1.33            |
|Багетная мастерская         |848   |1       |1      |1399096.0  |0.0      |1399096.0  |0.0             |
|РС (Распределительный склад)|249   |1       |9      |788680.0   |27859.0  |760821.0   |3.61            |
|Кафе-1 ( ул.Фадеева)        |3977  |2459    |2      |207762.0   |672.0    |207091.0   |0.33            |
+----------------------------+------+--------+

### 6.2. Динамика по месяцам

In [26]:
month_stats = (
    sales_c
    .groupBy("month")
    .agg(
        F.count("*").alias("lines"),
        F.round(F.sum("net_amount"), 0).alias("net_revenue"),
        F.round(F.sum(F.when(F.col("is_delivery"), F.col("net_amount")).otherwise(0)), 0).alias("delivery_revenue"),
    )
    .orderBy("month")
)
month_stats.show(30, truncate=False)

+-------+------+-----------+----------------+
|month  |lines |net_revenue|delivery_revenue|
+-------+------+-----------+----------------+
|2020-06|2932  |212505.0   |0.0             |
|2020-07|13218 |1109801.0  |0.0             |
|2020-08|11918 |1294099.0  |0.0             |
|2020-09|20421 |2121197.0  |57207.0         |
|2020-10|21995 |2664647.0  |342287.0        |
|2020-11|20728 |2243757.0  |54623.0         |
|2020-12|25501 |3386474.0  |411639.0        |
|2021-01|20506 |2555000.0  |187218.0        |
|2021-02|25085 |2972441.0  |223135.0        |
|2021-03|24446 |2829633.0  |215643.0        |
|2021-04|21864 |2541943.0  |0.0             |
|2021-05|29985 |3534045.0  |12750.0         |
|2021-06|32742 |4294454.0  |212529.0        |
|2021-07|61749 |1.1062055E7|2952225.0       |
|2021-08|69165 |1.1421666E7|2718096.0       |
|2021-09|120121|1.8618773E7|4533010.0       |
|2021-10|107459|1.7133593E7|3804411.0       |
|2021-11|108614|1.6951319E7|3473616.0       |
|2021-12|126773|2.2514605E7|531880

**Вывод.** Практически всю выручку дают три торговых зала: крупнейший ТЗ‑2 (≈58 млн после скидок), затем ТЗ‑1 (≈43 млн) и ТЗ‑3 (≈26 млн); кафе, багетная мастерская и распределительный склад малы. Средняя скидка невелика (0–3,6 %). Выручка резко растёт со второй половины 2021 г., а 2020‑06 — неполный первый месяц данных.

## 7. Соединения (JOIN)
### 7.1. Справочник товаров (LEFT JOIN)
Справочники читаем с явной схемой. Проверяем уникальность ключей до соединения, иначе JOIN размножит факты.

In [27]:
products_schema = StructType([
    StructField("product_key", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("product_group", StringType(), True),
])
clients_schema = StructType([
    StructField("client_id", StringType(), True),
    StructField("client_name", StringType(), True),
    StructField("contractor_name", StringType(), True),
])

products = (spark.read.option("header", True).option("sep", "\t")
            .schema(products_schema).csv(str(SRC / "Продукты.csv")))
clients_raw = (spark.read.option("header", True).option("sep", "\t")
               .schema(clients_schema).csv(str(SRC / "Клиенты.csv")))

print("products:", products.count(), "уникальных ключей:", products.select("product_key").distinct().count())
print("clients :", clients_raw.count(), "уникальных ключей:", clients_raw.select("client_id").distinct().count())

products: 21163 уникальных ключей: 21163
clients : 3838 уникальных ключей: 3227


В справочнике клиентов ключ `client_id` **не уникален** (один клиент может встречаться с несколькими «юрлицами»). Прямой JOIN размножил бы строки продаж. Оставляем одну запись на клиента (первую по `contractor_name`) — так связь становится «многие-к-одному».

In [28]:
clients_dupes = (clients_raw.groupBy("client_id").count().filter("count > 1"))
print("Клиентов с дублями:", clients_dupes.count())

clients = (
    clients_raw
    .groupBy("client_id")
    .agg(F.min("client_name").alias("client_name"))
)
print("clients после дедупликации:", clients.count())

n_before = sales_c.count()

sales_p = (
    sales_c.alias("s")
    .join(products.alias("p"), F.col("s.product_key") == F.col("p.product_key"), "left")
    .select("s.*", "p.product_name", "p.brand", "p.product_group")
    .withColumn("brand", F.coalesce("brand", F.lit("unknown")))
    .withColumn("group_l1", F.split("product_group", "-").getItem(0))
)
print("LEFT JOIN с товарами  до:", n_before, " после:", sales_p.count())
print("Строк без товара в справочнике:", sales_p.filter(F.col("product_name").isNull()).count())

Клиентов с дублями: 566
clients после дедупликации: 3227
LEFT JOIN с товарами  до: 865222  после: 865222
Строк без товара в справочнике: 0


### 7.2. Клиенты (INNER JOIN) и сравнение с LEFT JOIN

In [29]:
sales_full_inner = (
    sales_p.alias("s")
    .join(clients.alias("c"), F.col("s.client_id") == F.col("c.client_id"), "inner")
    .select("s.*", "c.client_name")
)
sales_full_left = (
    sales_p.alias("s")
    .join(clients.alias("c"), F.col("s.client_id") == F.col("c.client_id"), "left")
    .select("s.*", "c.client_name")
)
print("до JOIN        :", sales_p.count())
print("INNER JOIN     :", sales_full_inner.count())
print("LEFT JOIN      :", sales_full_left.count())
print("LEFT без клиента:", sales_full_left.filter(F.col("client_name").isNull()).count())

sales_full = sales_full_left

до JOIN        : 865222
INNER JOIN     : 865222
LEFT JOIN      : 865222
LEFT без клиента: 0


**Вывод.** Для фактовой витрины выбран LEFT JOIN: он сохраняет все продажи и позволяет измерить неполноту справочников. INNER JOIN здесь даёт тот же результат, потому что все ключи находятся в справочниках (проверено выше), и он допустим как контроль целостности. Дедупликация справочника клиентов предотвратила размножение строк.

## 8. Spark SQL

In [30]:
sales_full.createOrReplaceTempView("sales_full")

# Запрос 1: топ товарных групп по чистой выручке (GROUP BY + сортировка по агрегату)
top_groups = spark.sql("""
SELECT
    group_l1,
    COUNT(*)                          AS lines,
    COUNT(DISTINCT client_id)         AS clients,
    ROUND(SUM(net_amount), 0)         AS net_revenue,
    ROUND(SUM(discount) / SUM(amount) * 100, 2) AS discount_pct
FROM sales_full
GROUP BY group_l1
ORDER BY net_revenue DESC
LIMIT 10
""")
top_groups.show(truncate=False)

# Запрос 2: топ клиентов и доля доставки
top_clients = spark.sql("""
SELECT
    client_id,
    client_name,
    COUNT(*)                                   AS lines,
    ROUND(SUM(net_amount), 0)                  AS net_revenue,
    ROUND(SUM(CASE WHEN is_delivery THEN net_amount ELSE 0 END) / SUM(net_amount) * 100, 1) AS delivery_share_pct
FROM sales_full
GROUP BY client_id, client_name
HAVING SUM(net_amount) > 0
ORDER BY net_revenue DESC
LIMIT 10
""")
top_clients.show(truncate=False)

# Запрос 3: маржа по бренду только для строк с известной себестоимостью
top_brands = spark.sql("""
SELECT
    brand,
    COUNT(*)                   AS lines,
    ROUND(SUM(net_amount), 0)  AS net_revenue,
    ROUND(SUM(margin), 0)      AS margin,
    ROUND(SUM(margin) / SUM(net_amount) * 100, 1) AS margin_pct
FROM sales_full
WHERE cost_price IS NOT NULL AND quantity IS NOT NULL AND net_amount > 0
GROUP BY brand
HAVING COUNT(*) >= 500
ORDER BY margin DESC
LIMIT 10
""")
top_brands.show(truncate=False)

+---------------------+------+-------+-----------+------------+
|group_l1             |lines |clients|net_revenue|discount_pct|
+---------------------+------+-------+-----------+------------+
|Живопись             |182227|1224   |2.5446236E7|2.0         |
|Графические материалы|157061|1001   |1.4019454E7|1.53        |
|ДПИ                  |95303 |720    |1.309835E7 |1.59        |
|Кисти                |111110|866    |1.0385336E7|1.57        |
|Основы               |22644 |446    |9325080.0  |2.45        |
|Мольберты            |2653  |61     |7681934.0  |1.35        |
|z                    |60456 |667    |6267467.0  |1.77        |
|Бумага               |30346 |488    |5856805.0  |1.67        |
|Подрамники, рамы     |14269 |279    |4755726.0  |1.96        |
|Литература           |12772 |202    |4098751.0  |1.44        |
+---------------------+------+-------+-----------+------------+

+---------+-------------------------------------------------+------+-----------+------------------+
|cl

**Эквивалент через DataFrame API** (проверка совпадения результатов SQL и API):

In [31]:
top_groups_api = (
    sales_full.groupBy("group_l1")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("client_id").alias("clients"),
        F.round(F.sum("net_amount"), 0).alias("net_revenue"),
        F.round(F.sum("discount") / F.sum("amount") * 100, 2).alias("discount_pct"),
    )
    .orderBy(F.desc("net_revenue"))
    .limit(10)
)
print("SQL == DataFrame API:", top_groups.collect() == top_groups_api.collect())

SQL == DataFrame API: True


**Вывод.** SQL и DataFrame API дают одинаковый результат. Запросы ранжируют товарные группы, клиентов и бренды; рейтинг по марже считается только по строкам с известной себестоимостью.

## 9. Итоговая витрина и экспорт
Витрина: выручка, скидки и маржа по связке «месяц × филиал × товарная группа верхнего уровня».

In [32]:
mart = (
    sales_full
    .groupBy("month", "branch", "group_l1")
    .agg(
        F.count("*").alias("lines"),
        F.round(F.sum("quantity"), 2).alias("qty"),
        F.round(F.sum("amount"), 2).alias("gross"),
        F.round(F.sum("discount"), 2).alias("discount"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue"),
        F.round(F.sum("margin"), 2).alias("margin_known"),
    )
    .orderBy("month", "branch", F.desc("net_revenue"))
)
mart_rows = mart.count()
print("Строк в витрине:", mart_rows)
mart.printSchema()

OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

mart.write.mode("overwrite").parquet(str(OUT / "own_sales_mart_parquet"))

(
    branch_stats
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(OUT / "own_branch_stats_csv"))
)
print(sorted(p.name for p in (OUT / "own_branch_stats_csv").iterdir() if not p.name.startswith(".")))

Строк в витрине: 1780
root
 |-- month: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- group_l1: string (nullable = true)
 |-- lines: long (nullable = false)
 |-- qty: double (nullable = true)
 |-- gross: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- net_revenue: double (nullable = true)
 |-- margin_known: double (nullable = true)

['_SUCCESS', 'part-00000-9214b6a9-f61d-4fbd-81d7-0738eda826d4-c000.csv']


### Повторное чтение Parquet (round-trip)

In [33]:
mart_back = spark.read.parquet(str(OUT / "own_sales_mart_parquet"))

print("Строк до записи :", mart_rows)
print("Строк после чтения:", mart_back.count())
same = [(f.name, f.dataType) for f in mart.schema] == [(f.name, f.dataType) for f in mart_back.schema]
print("Имена и типы столбцов совпадают:", same, "(Parquet делает все поля nullable)")
mart_back.printSchema()

a = mart.agg(F.sum("net_revenue")).first()[0]
b = mart_back.agg(F.sum("net_revenue")).first()[0]
print("Сумма net_revenue до/после:", a, b, "равны:", abs(a - b) < 1e-6)

Строк до записи : 1780
Строк после чтения: 1780
Имена и типы столбцов совпадают: True (Parquet делает все поля nullable)
root
 |-- month: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- group_l1: string (nullable = true)
 |-- lines: long (nullable = true)
 |-- qty: double (nullable = true)
 |-- gross: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- net_revenue: double (nullable = true)
 |-- margin_known: double (nullable = true)

Сумма net_revenue до/после: 129462006.96999995 129462006.96999995 равны: True


Завершаем сессию (последняя ячейка pipeline).

In [34]:
spark.stop()

## 10. Итоговые выводы
1. **Структура продаж.** Розничный покупатель «Частное лицо» (`CL_25101`) даёт ≈96 % строк и ≈102 млн из ≈129 млн чистой выручки; остальные клиенты — юрлица с доставкой (доля доставки у них 100 %).
2. **Филиалы.** Три торговых зала формируют почти всю выручку; ТЗ‑2 крупнейший (≈58 млн), но у ТЗ‑1 самая высокая средняя скидка (1,35 %) и больше всего клиентов-юрлиц (1 469).
3. **Динамика.** Выручка растёт: с ≈2–3 млн в месяц в 2020 — первой половине 2021 до ≈17–22 млн в сентябре—декабре 2021; доля выручки с доставкой в конце 2021 г. около 20–27 %.
4. **Товары.** Крупнейшая корневая группа — «Живопись» (≈25 млн, скидка 2,0 %), далее «Графические материалы» и «ДПИ»; наибольшая маржа по брендам — у издательства АСТ и «Передвижника».
5. **Качество данных.** Пропуски структурны: адрес отсутствует при самовывозе, чек — у оптовых/мастерских отгрузок, себестоимость неизвестна у ≈10 % строк. Справочник клиентов содержал 566 дублирующихся ключей — без дедупликации JOIN размножил бы факты; после неё число строк сохранилось (865 222 до и после LEFT/INNER JOIN).
6. **Spark.** SQL и DataFrame API дали одинаковые результаты; витрина (1 780 строк) записана в Parquet и прочитана обратно без потери строк и суммы.

**Ограничения анализа.**
* Расчёт маржи опирается на допущение, что `Себестоимость` — за единицу товара; у ≈13 % строк с известной себестоимостью маржа отрицательна (продажа ниже себестоимости либо неверное допущение), а у ≈10 % строк себестоимость неизвестна вовсе, поэтому маржа ориентировочная.
* Данные охватывают неполный период (с 2020‑06, основная масса — с 2021‑07), а розничный клиент агрегирован в одну запись, поэтому нельзя судить о поведении отдельных покупателей; результаты описывают выборку и не доказывают причинных связей.